Data Reading

In [1]:
import pandas as pd

df=pd.read_parquet('c:\\Users\\Utkarsh\\Desktop\\Azure DataBricks Project\\Sales-Analytics-Using-Azure-Databricks\\data\\processed\\sales_cleaned.parquet')

In [2]:
df.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,MRP_Category
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380,Premium
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228,Low
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700,Medium
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,Unknown,Tier 3,Grocery Store,732.3800,High
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052,Low


DuckDB

In [8]:
import duckdb

duckdb.sql("Select * from df limit 5").df()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,MRP_Category
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380,Premium
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228,Low
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700,Medium
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,Unknown,Tier 3,Grocery Store,732.3800,High
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052,Low


Where CLause

In [11]:
duckdb.sql("""Select * from df 
           where Item_Visibility=0
           Limit 5""").df()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,MRP_Category
0,FDX07,19.200,Regular,0.0,Fruits and Vegetables,182.0950,OUT010,1998,Unknown,Tier 3,Grocery Store,732.3800,High
1,NCD19,8.930,Low Fat,0.0,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052,Low
2,FDP36,10.395,Regular,0.0,Baking Goods,51.4008,OUT018,2009,Medium,Tier 3,Supermarket Type2,556.6088,Low
3,FDY07,11.800,Low Fat,0.0,Fruits and Vegetables,45.5402,OUT049,1999,Medium,Tier 1,Supermarket Type1,1516.0266,Low
4,FDP33,18.700,Low Fat,0.0,Snack Foods,256.6672,OUT018,2009,Medium,Tier 3,Supermarket Type2,3068.0064,Premium


Indentifying the Pattern of Zero-Item Visibilty 

In [23]:
duckdb.sql("""Select
                Case
                    When Item_Visibility=0 then 'Zero_visibility'
                    else 'Non_Zer_Visibility' 
               

                End as Visibility_Status,

                Count (*) Records,

                from df

                Group by Visibility_Status
                """).df()

,Visibility_Status,Records
0,Zero_visibility,526
1,Non_Zer_Visibility,7993


Group By and Order By

In [15]:
duckdb.sql("""Select sum(Item_Outlet_Sales) as Total_Outlet_Sales, MRP_Category from df 
           where Item_Visibility=0
           Group by MRP_Category
           order by Total_Outlet_Sales desc""").df()

,Total_Outlet_Sales,MRP_Category
0,440429.3632,Premium
1,348693.4418,High
2,247488.5128,Medium
3,132446.2624,Low


Records with zero recorded vsibility havs still large size of sales

In [19]:
duckdb.sql("""Select 
            Case
                when Item_Visibility = 0 Then 'Zero Visibility'
                else 'Non-Zero Visibility'
            End as Visibility_Status,

            Count (*) as Records,

            sum (Item_Outlet_Sales) as Total_Sales,

            avg(Item_Outlet_Sales) as Average_Sales,

        from df

        Group by Visibility_Status
            """).df()

,Visibility_Status,Records,Total_Sales,Average_Sales
0,Non-Zero Visibility,7993,1.741249e+07,2178.467363
1,Zero Visibility,526,1.169058e+06,2222.542928


In [24]:
duckdb.sql("""Select 
            sum(Item_Outlet_Sales)

        from df

        
            """).df()

,sum(Item_Outlet_Sales)
0,1.858155e+07


Observation:

The Total records of the Zero visibility is 6.17%.

The Total Sales/Revenue is 6.3%

But

The Average Sales is slightly higher than Non-Zero Visibility.

Need to analyse the average sales of zero visibility records consistent across different categories and outlets?

In [27]:
duckdb.sql("""Select 
            MRP_Category,
            Case
                when Item_Visibility = 0 Then 'Zero Visibility'
                else 'Non-Zero Visibility'
            End as Visibility_Status,

            Count (*) as Records,


            avg(Item_Outlet_Sales) as Average_Sales,

        from df

        Group by MRP_Category, Visibility_Status
        Order by MRP_Category, Visibility_Status
            """).df()

,MRP_Category,Visibility_Status,Records,Average_Sales
0,Low,Non-Zero Visibility,1987,949.068581
1,Low,Zero Visibility,143,926.197639
2,Medium,Non-Zero Visibility,1998,1749.261872
3,Medium,Zero Visibility,132,1874.912976
4,High,Non-Zero Visibility,2006,2560.176909
5,High,Zero Visibility,123,2834.906031
6,Premium,Non-Zero Visibility,2002,3444.530600
7,Premium,Zero Visibility,128,3440.854400


In Medium & High Cat the Avg Sales difference between Zero and Non-Zero, where Zero visibility has higher sales.

The Premium Cat performs equal

The Low Cat has high Non-Zero visibility Avg Sales.


So, the early findings of zero visibility has slightly higher average is becoming **True**, but need more information now why it is happening?

In [28]:
duckdb.sql("""Select 
            MRP_Category,
            Outlet_Type,
            Case
                when Item_Visibility = 0 Then 'Zero Visibility'
                else 'Non-Zero Visibility'
            End as Visibility_Status,

            Count (*) as Records,


            avg(Item_Outlet_Sales) as Average_Sales,

        from df

        Group by MRP_Category,Outlet_type ,Visibility_Status
        Order by MRP_Category, Outlet_Type, Visibility_Status
            """).df()

,MRP_Category,Outlet_Type,Visibility_Status,Records,Average_Sales
0,Low,Grocery Store,Non-Zero Visibility,253,150.312903
1,Low,Grocery Store,Zero Visibility,22,152.619518
2,Low,Supermarket Type1,Non-Zero Visibility,1304,995.344450
3,Low,Supermarket Type1,Zero Visibility,83,960.797530
4,Low,Supermarket Type2,Non-Zero Visibility,209,806.557766
5,Low,Supermarket Type2,Zero Visibility,20,921.167590
6,Low,Supermarket Type3,Non-Zero Visibility,221,1725.205294
7,Low,Supermarket Type3,Zero Visibility,18,1717.727011
8,Medium,Grocery Store,Non-Zero Visibility,251,267.174134
9,Medium,Grocery Store,Zero Visibility,9,214.461578


Observation:

In segmentation by MRP category and Outlet type displayed no consistent relationship.

The difference vary across outlet types and MRP segements, but the largest observation difference occured in High MRP products in Supermarket Type 3. However this segement contains only 14 zero visibility records, so the result should be treated cautiously.

Which Products and Outlets are responsible for those higher average sales having zero visibility?

In [29]:
duckdb.sql("""Select 
            
           Item_Identifier,
           Item_Type,
           Item_MRP,
           Item_Visibility,
           Item_Outlet_Sales,
           Outlet_Identifier,
           Outlet_Type

        from df

        where MRP_Category = 'High'
        and Outlet_Type = 'Supermarket Type3'
        and Item_Visibility = 0

        order by Item_Outlet_Sales desc

        
            """).df()

,Item_Identifier,Item_Type,Item_MRP,Item_Visibility,Item_Outlet_Sales,Outlet_Identifier,Outlet_Type
0,FDU56,Fruits and Vegetables,184.7266,0.0,8114.7704,OUT027,Supermarket Type3
1,DRE49,Soft Drinks,153.8024,0.0,7741.9224,OUT027,Supermarket Type3
2,FDY39,Meat,182.0608,0.0,7717.9536,OUT027,Supermarket Type3
3,NCL53,Health and Hygiene,175.4028,0.0,6729.9064,OUT027,Supermarket Type3
4,FDV01,Canned,154.6314,0.0,6515.5188,OUT027,Supermarket Type3
5,DRM23,Hard Drinks,171.7422,0.0,6207.9192,OUT027,Supermarket Type3
6,FDZ50,Dairy,184.2608,0.0,6064.1064,OUT027,Supermarket Type3
7,FDE36,Baking Goods,165.5868,0.0,5404.9644,OUT027,Supermarket Type3
8,FDS16,Frozen Foods,145.2760,0.0,5273.1360,OUT027,Supermarket Type3
9,FDQ33,Snack Foods,151.6708,0.0,4815.0656,OUT027,Supermarket Type3


Since there where only 14 zero records in the the strongest difference between Category and Outlet type.

The result shows the Outlet OUT027 has all 14 zero visibility records.



Does OUT027 generally have Higher Sales, regardless of visibility?



In [30]:
duckdb.sql("""Select 
            Outlet_Identifier,
            Case
                when Item_Visibility = 0 Then 'Zero Visibility'
                else 'Non-Zero Visibility'
            End as Visibility_Status,

            Count (*) as Records,

            sum (Item_Outlet_Sales) as Total_Sales,
            avg(Item_Outlet_Sales) as Average_Sales

        from df
        where Outlet_Identifier = 'OUT027'
        Group by Outlet_Identifier, Visibility_Status
        Order by Average_Sales desc;
            """).df()

,Outlet_Identifier,Visibility_Status,Records,Total_Sales,Average_Sales
0,OUT027,Zero Visibility,60,2.219498e+05,3699.162607
1,OUT027,Non-Zero Visibility,872,3.222519e+06,3695.548860


Observation:

The Average Sales has no difference between Zero and Non-Zero visibility in OUT027.


In the above findings the High MRP + Supermarket Type 3 + Zero V shows high average sales with 14 records which belongs to OUT027.

But in later analysis we got to know there is no difference between Zero and Non-zero visibility average sales for OUT027.




Does OUT027 generally have higher sales than other outlets?

In [31]:
duckdb.sql("""Select 
            
           
           Outlet_Identifier,
            count (*) as Records,
            sum (Item_Outlet_Sales) as Total_Sales,
            avg(Item_Outlet_Sales) as Average_Sales

        from df
        group by Outlet_Identifier
        order by Average_Sales desc;

        
            """).df()

,Outlet_Identifier,Records,Total_Sales,Average_Sales
0,OUT027,932,3.444468e+06,3695.781505
1,OUT035,930,2.268123e+06,2438.841866
2,OUT049,930,2.183970e+06,2348.354635
3,OUT017,926,2.167465e+06,2340.675263
4,OUT013,932,2.142664e+06,2298.995256
5,OUT046,930,2.118395e+06,2277.844267
6,OUT045,929,2.036725e+06,2192.384798
7,OUT018,928,1.851823e+06,1995.498739
8,OUT019,527,1.795736e+05,340.746838
9,OUT010,555,1.883402e+05,339.351662


Observation:

OUT027 has the highest average sales per product-outlet record among the outlets in this dataset.

OUT027's average is approximately 51.5% higher than OUT035's.

The OUT027 has highest Total Sales and Average Sales which is making high-performing outlet in the dataset.

However, within OUT027, zero and non-zero visibility records have almost identical average sales.

This suggests that OUT027's overall high sales performance is unlikely to be explained by visibility status alone.

Till now:

Question:

Does zero visibility behave differently from non-zero visibility in terms of sales?

=> Zero visibility records can still have sales it does not mean zero sales.
The zero visibility is neither valid nor invalid.

## Zero's in particular Outlets

In [32]:
duckdb.sql("""Select 
            
           
           Outlet_Identifier,
            count (*) as Zero_Visibility_Records

        from df
        where Item_Visibility = 0
        group by Outlet_Identifier
        order by Zero_Visibility_Records desc;

        
            """).df()

,Outlet_Identifier,Zero_Visibility_Records
0,OUT018,65
1,OUT046,61
2,OUT027,60
3,OUT013,59
4,OUT045,58
5,OUT017,57
6,OUT035,54
7,OUT049,53
8,OUT019,30
9,OUT010,29


In [ ]:
duckdb.sql("""Select 
            
           
           Outlet_Identifier,
            count (*) as Total_Records,
            count (Case when Item_Visibility = 0 then 1 end) as Zero_Visibility_Records

        from df

        group by Outlet_Identifier
        order by Zero_Visibility_Records desc;

        
            """).df()

,Outlet_Identifier,Total_Records,Zero_Visibility_Records
0,OUT018,928,65
1,OUT046,930,61
2,OUT027,932,60
3,OUT013,932,59
4,OUT045,929,58
5,OUT017,926,57
6,OUT035,930,54
7,OUT049,930,53
8,OUT019,527,30
9,OUT010,555,29


In [35]:
duckdb.sql("""Select 
            
           
           Outlet_Identifier,
            count (*) as Total_Records,
            count (Case when Item_Visibility = 0 then 1 end) as Zero_Visibility_Records,

            round((count (Case when Item_Visibility = 0 then 1 end)/count (*))*100,2) as Zero_Visibility_Percentage

        from df

        group by Outlet_Identifier
        order by Zero_Visibility_Records desc;

        
            """).df()

,Outlet_Identifier,Total_Records,Zero_Visibility_Records,Zero_Visibility_Percentage
0,OUT018,928,65,7.00
1,OUT046,930,61,6.56
2,OUT027,932,60,6.44
3,OUT013,932,59,6.33
4,OUT045,929,58,6.24
5,OUT017,926,57,6.16
6,OUT035,930,54,5.81
7,OUT049,930,53,5.70
8,OUT019,527,30,5.69
9,OUT010,555,29,5.23


Observation:

The Zeros number range from 29 to 65 have closer together roughly 5 to 7%.

The observation suggest that the zeros are not overwhelmingly concentrated in one particular outlet.
Instead, the zeros appear to occur across multiple outlets at fairly similar rates.

The overall percenatge of zero records is 6.17%

The Lowest total record outlet (OUT010)shows 5.2% bleow the overall rate
The highest total recors outlet(OUT018) shows 7% above the overall rate
The highest avg sales outlet (OUT027) shows 6.4% close to overall rate.


So, the zeros and not conentrated in one outlet.

There is no strong evidence that the zeros are an outlet-specific data-quality problem.

In [37]:
duckdb.sql("""Select 
            
           
           Item_type,
            count (*) as Total_Records,
            count (Case when Item_Visibility = 0 then 1 end) as Zero_Visibility_Records,

            round((count (Case when Item_Visibility = 0 then 1 end)/count (*))*100,2) as Zero_Visibility_Percentage

        from df

        group by Item_type
        order by Zero_Visibility_Records desc;

        
            """).df()

,Item_Type,Total_Records,Zero_Visibility_Records,Zero_Visibility_Percentage
0,Fruits and Vegetables,1232,87,7.06
1,Snack Foods,1199,68,5.67
2,Frozen Foods,855,51,5.96
3,Household,910,51,5.60
4,Baking Goods,647,45,6.96
5,Dairy,681,38,5.58
6,Soft Drinks,445,34,7.64
7,Health and Hygiene,520,33,6.35
8,Canned,649,33,5.08
9,Meat,425,20,4.71


Starchy Foods has the highest zero visibility 10.14%, compared with approximately 6.17% overall although contain only 148 records.


Most of the catrgories are round 5 to 8% apart from starchy foods.

## Relationship between Zero Visibility and the actual sales behavious at the item level

In [38]:
duckdb.sql("""Select 
           
            Case
                when Item_Visibility = 0 Then 'Zero Visibility'
                else 'Non-Zero Visibility'
            End as Visibility_Status,

            Count (*) as Records,

            avg (Item_MRP) as Average_MRP,
            avg(Item_Outlet_Sales) as Average_Sales,
            min(Item_Outlet_Sales) as Minimum_Sales,
            max(Item_Outlet_Sales) as Maximum_Sales

        from df
        
        Group by Visibility_Status
    
            """).df()

,Visibility_Status,Records,Average_MRP,Average_Sales,Minimum_Sales,Maximum_Sales
0,Non-Zero Visibility,7993,141.200392,2178.467363,33.29,13086.9648
1,Zero Visibility,526,138.117157,2222.542928,33.29,11445.1020


Observation:

The zero visibility group doesn't have a higher average MRP.

The zero visibility group has approximately 2% higher average sales.

This is an observed association, not evidence that zero visibility produces higher sales.


The average could be high because of one large value.

### Quartiles Analysis

In [39]:
duckdb.sql("""Select 
           
            Case
                when Item_Visibility = 0 Then 'Zero Visibility'
                else 'Non-Zero Visibility'
            End as Visibility_Status,

            Count (*) as Records,

            avg(Item_Outlet_Sales) as Average_Sales,
            median(Item_Outlet_Sales) as Median_Sales,
            quantile(Item_Outlet_Sales, 0.25) as First_Quartile,
            quantile(Item_Outlet_Sales, 0.75) as Third_Quartile,
            min(Item_Outlet_Sales) as Minimum_Sales,
            max(Item_Outlet_Sales) as Maximum_Sales

        from df
        
        Group by Visibility_Status
    
            """).df()

,Visibility_Status,Records,Average_Sales,Median_Sales,First_Quartile,Third_Quartile,Minimum_Sales,Maximum_Sales
0,Zero Visibility,526,2222.542928,1774.0241,890.8404,3185.1872,33.29,11445.1020
1,Non-Zero Visibility,7993,2178.467363,1794.3310,829.5868,3095.9700,33.29,13086.9648


Observation:

The zero visibility group has a higher average, but its median is actually slightly lower.

The zero visibility sales distribution is right-skewed some relatively high-sales records are pulling the average upward.

The zero-visibility group has a slightly higher Q1 and Q3.
So there is some evidence that the distribution of the zero visibility group is somewhat higher in the middle portions, but median is lower.

Therefore, the higher mean should not be interpreted as evidence that zero-visibility products generally achieve higher sales.